# Airfare Route Segmentation by Historical Pricing Behaviour
## Phase 2: Unsupervised Clustering, Metric Evaluation & PCA Visualization
**Objective**:
1. Apply K-Means across at least three cluster counts ($K=3, 4, 5$).
2. Compare Euclidean ($L_2$) and Manhattan ($L_1$) distance metrics.
3. Perform PCA dimensionality reduction (Explained Variance & Loadings).
4. Evaluate clustering solutions using WCSS (Inertia), Silhouette Score, and Davies-Bouldin Index.
5. Select a defensible clustering structure.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../src"))
from data_loader import load_or_generate_dataset
from feature_engineering import preprocess_flight_records, aggregate_route_features, prepare_clustering_matrix
from clustering import run_kmeans_clustering, run_manhattan_clustering, compute_pca_projection
from evaluation import evaluate_kmeans_sweep, compare_distance_metrics, evaluate_cluster_stability

# Prepare data
df = load_or_generate_dataset()
clean_df = preprocess_flight_records(df)
route_df = aggregate_route_features(clean_df)
X_scaled, scaler, feature_names = prepare_clustering_matrix(route_df)


### 1. Multi-K Evaluation Sweep: WCSS, Silhouette & Davies-Bouldin
We evaluate K-Means across $K \in [2, 8]$ to determine the optimal clustering partition.


In [ ]:
k_eval_df = evaluate_kmeans_sweep(X_scaled, k_range=[2, 3, 4, 5, 6, 7, 8])
print(k_eval_df.to_string(index=False))


### 2. Distance Metric Benchmark: Euclidean ($L_2$) vs Manhattan ($L_1$)
**Theoretical Justification**:
Airfare time-series exhibit extreme positive price spikes (e.g. emergency bookings, holiday peak fares).
- **Euclidean Distance** ($L_2$ norm): Squares errors ($\sum (x_i - \mu_i)^2$), causing cluster centroids to be pulled disproportionately toward extreme outliers.
- **Manhattan Distance** ($L_1$ norm): Evaluates absolute differences ($\sum |x_i - c_i|$), yielding robust medians that resist leverage from price spikes.


In [ ]:
dist_comparison_df = compare_distance_metrics(X_scaled, k_values=[3, 4, 5])
print(dist_comparison_df.to_string(index=False))


### 3. Principal Component Analysis (PCA) Dimensionality Reduction
We project the 9-dimensional route behavior space onto principal components.


In [ ]:
pca, X_pca, exp_var = compute_pca_projection(X_scaled, n_components=3)
print(f"Explained Variance Ratio:")
print(f"  PC1: {exp_var[0]*100:.2f}%")
print(f"  PC2: {exp_var[1]*100:.2f}%")
print(f"  PC3: {exp_var[2]*100:.2f}%")
print(f"Total 3D Variance Captured: {np.sum(exp_var)*100:.2f}%
")

loadings_df = pd.DataFrame(pca.components_.T, index=feature_names, columns=['PC1', 'PC2', 'PC3'])
print("PCA Loadings Matrix:")
print(loadings_df.round(3))


### 4. Defensible Structure Selection ($K=4$) & Stability Verification
We assess cluster stability across 10 random seed initializations using the **Adjusted Rand Index (ARI)** and **Normalized Mutual Information (NMI)**.


In [ ]:
stability = evaluate_cluster_stability(X_scaled, k=4, n_runs=10)
print(f"Stability Results for K=4:")
print(f"  Mean Adjusted Rand Index (ARI): {stability['mean_ari']} (+/- {stability['std_ari']})")
print(f"  Mean Normalized Mutual Info (NMI): {stability['mean_nmi']} (+/- {stability['std_nmi']})")
print("An ARI of 1.0 indicates perfect deterministic stability across varying seed initializations.")
